# Day 3 — LightGBM with engineered features

Beat the persistence-24h baseline (MAPE 4.5%) using gradient boosting on engineered features:
lag/rolling, calendar (incl. US holidays + cyclical encoding), and **lagged** Houston weather.

Also fit quantile models (q=0.1, 0.5, 0.9) for prediction intervals — coverage analysis on Day 7.

## Setup

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from forecasting.data import load_demand, load_weather
from forecasting.backtest import split_train_val_test, TRAIN_END, VAL_END
from forecasting.features import build_features, FEATURE_COLS, TARGET
from forecasting.metrics import summarize

sns.set_theme(style='whitegrid', context='notebook')
pd.options.display.float_format = '{:,.3f}'.format

## Build features

In [ ]:
demand = load_demand('ERCO')
weather = load_weather('USW00012960')   # Houston Hobby

feats = build_features(demand, weather, horizon_hours=24)
print(f'features: {len(feats):,} rows × {len(feats.columns)} cols')
feats.head(3)

In [ ]:
# Drop rows with missing lags / rolling stats (early-train period)
feats_clean = feats.dropna(subset=FEATURE_COLS + [TARGET])
print(f'after dropna: {len(feats_clean):,} rows')

# Temporal split
train, val, test = split_train_val_test(feats_clean, ts_col='ts')
for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f'{name:5s} {len(part):>6,} rows  ({part.ts.min()} → {part.ts.max()})')

## Point forecast (q=0.5)

In [ ]:
def make_xy(df):
    return df[FEATURE_COLS], df[TARGET]

X_train, y_train = make_xy(train)
X_val,   y_val   = make_xy(val)
X_test,  y_test  = make_xy(test)

params = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=50,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    verbose=-1,
)

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50)],
)
y_pred = model.predict(X_test)
summarize(y_test.to_numpy(), y_pred)

## Quantile models (0.1, 0.5, 0.9) for prediction intervals

In [ ]:
from forecasting.metrics import coverage, pinball_loss

quantile_preds = {}
for q in (0.1, 0.5, 0.9):
    qm = lgb.LGBMRegressor(objective='quantile', alpha=q, **params)
    qm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50)])
    quantile_preds[q] = qm.predict(X_test)

p10, p50, p90 = quantile_preds[0.1], quantile_preds[0.5], quantile_preds[0.9]

print(f'median q50 metrics: {summarize(y_test.to_numpy(), p50)}')
print(f'80% PI coverage: {coverage(y_test.to_numpy(), p10, p90):.3f}  (target 0.80)')
print(f'pinball loss q10/q50/q90: '
      f'{pinball_loss(y_test.to_numpy(), p10, 0.1):.1f} / '
      f'{pinball_loss(y_test.to_numpy(), p50, 0.5):.1f} / '
      f'{pinball_loss(y_test.to_numpy(), p90, 0.9):.1f}')

## Update scoreboard

In [ ]:
scoreboard_path = Path('..') / 'results' / 'scoreboard.json'
scoreboard = json.loads(scoreboard_path.read_text()) if scoreboard_path.exists() else {'results': {}}

scoreboard['results']['lightgbm_point'] = {k: float(v) for k, v in summarize(y_test.to_numpy(), y_pred).items()}
scoreboard['results']['lightgbm_q50'] = {k: float(v) for k, v in summarize(y_test.to_numpy(), p50).items()}
scoreboard['results']['lightgbm_q50']['coverage_80pi'] = float(coverage(y_test.to_numpy(), p10, p90))

scoreboard_path.write_text(json.dumps(scoreboard, indent=2))
print(f'Updated {scoreboard_path}')

## Feature importance

In [ ]:
imp = pd.DataFrame({'feature': FEATURE_COLS, 'importance': model.feature_importances_}) \
        .sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(data=imp, y='feature', x='importance', ax=ax, color='steelblue')
ax.set_title('LightGBM feature importance')
plt.tight_layout(); plt.show()

## Visualize one week

In [ ]:
vis = test.copy().reset_index(drop=True)
vis['y_pred'] = y_pred
vis['p10'] = p10; vis['p50'] = p50; vis['p90'] = p90
wk = vis[(vis.ts >= '2024-08-05') & (vis.ts < '2024-08-12')]

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(wk.ts, wk.p10, wk.p90, alpha=0.2, label='80% PI')
ax.plot(wk.ts, wk.demand_mwh, label='actual', color='black', lw=2)
ax.plot(wk.ts, wk.y_pred, label='LightGBM point', alpha=0.8)
ax.plot(wk.ts, wk.p50, label='LightGBM q50', alpha=0.5, ls='--')
ax.legend(); ax.set_title('LightGBM vs actual — first full week of August 2024'); ax.set_ylabel('Demand (MWh)')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## Takeaways for Day 5-6 (Chronos)

- LightGBM should beat persistence-24h's 4.5% MAPE — by how much depends on whether weather + calendar features add real signal.
- Watch the feature importance plot: typically `lag_24h`, `rolling_24h_mean`, and `tmax_c_lag1` dominate.
- 80% prediction interval coverage should be close to 0.80. Under-coverage = overconfident; over-coverage = wasteful intervals.
- Chronos on Day 5-6 has to beat this — and it does so zero-shot if it does, which is the headline story.